In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, log_loss, roc_auc_score



from agg import build_all_channel_aggregates, compute_channel_shares, compute_channel_diversity, compute_liquidity_signals

In [24]:
df = pd.read_csv("Train.csv")
out = build_all_channel_aggregates(df)
out.head(2)

,paybill_total_volume,paybill_total_value,paybill_avg_monthly_value,paybill_max_single_txn,paybill_active_months,paybill_avg_unique_cparty,paybill_value_volatility,merchantpay_total_volume,merchantpay_total_value,merchantpay_avg_monthly_value,...,received_active_months,received_avg_unique_cparty,received_value_volatility,deposit_total_volume,deposit_total_value,deposit_avg_monthly_value,deposit_max_single_txn,deposit_active_months,deposit_avg_unique_cparty,deposit_value_volatility
0,28,108361.61,18060.268333,5352.41,4,4.50,1.273622,7,11856.69,1976.115000,...,5,7.4,0.911386,23,266964.00,44494.000000,16608.05,2,4.5,1.821375
1,25,112041.45,18673.575000,5233.36,4,2.75,1.683330,15,25947.64,4324.606667,...,5,11.0,0.682034,71,915309.82,152551.636667,17082.32,5,2.0,0.824152


In [25]:
# shares = compute_channel_shares(out)
# shares.head(5)
# Concatinate it with out
# out = pd.concat([out, shares], axis=1)

In [26]:
# diversity_out = compute_channel_diversity(out, direction= "outflow")
# out['outflow_channel_entropy'] = diversity_out

In [27]:
sig = compute_liquidity_signals(df, out)
# out = pd.concat([out, sig], axis=1)

In [28]:
print(set(sig.columns) - set(out.columns))
in_both = set(sig.columns).intersection(set(out.columns))
# Remove duplicates before concat
out.drop(columns=in_both, inplace=True)
out = pd.concat([out, sig], axis=1)

{'total_inflow_value', 'paybill_share_trend', 'outflow_channel_entropy', 'total_outflow_value', 'withdraw_share_trend', 'paybill_share_of_outflow', 'outflow_to_inflow_ratio', 'balance_volatility', 'balance_trend', 'withdraw_share_of_outflow', 'avg_balance_6m'}


In [32]:
# Create new dataframe with original df categorical features and aggregated features

TARGET = "liquidity_stress_next_30d"
# df_cat = df.select_dtypes(exclude=["int64", "float64"])
new_df = pd.concat([df.drop(columns=["ID", TARGET]), df[TARGET], out], axis=1)


In [33]:
new_df.head(2)

,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,...,withdraw_share_of_outflow,paybill_share_of_outflow,total_outflow_value,total_inflow_value,outflow_to_inflow_ratio,outflow_channel_entropy,withdraw_active_months,paybill_active_months,withdraw_share_trend,paybill_share_trend
0,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,8,58372.55,...,0.498137,0.325910,332489.28,590474.93,0.563088,1.106951,1,4,0.622915,-0.246824
1,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,0,0.00,...,0.732322,0.126699,884314.06,1286319.71,0.687476,0.838202,6,4,-0.205591,0.143456


In [82]:
# Encode categorical features, do train-test split, and train model (from scratch - lightgbm)
# Encode categorical features
encoded_df = pd.get_dummies(new_df)

# Train-test split
X = encoded_df.drop(columns=[TARGET])
y = encoded_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
param_grid = {
    "scale_pos_weight": 5,
    "max_depth": 3,
    "num_leaves": 8,
    "colsample_bytree": 0.5,
    # "reg_alpha": 10,
    # "reg_lambda": 20,
    "learning_rate":0.01,
    "n_estimators": 5000,
    "importance_type": 'gain'
}

model = LGBMClassifier(**param_grid, random_state=42, n_jobs=4, verbosity=-1, eval_metric="logloss")
model.fit(X_train, y_train)


,num_leaves,8
,max_depth,3
,learning_rate,0.01
,n_estimators,5000
,colsample_bytree,0.5
,random_state,42
,n_jobs,4
,importance_type,'gain'
,scale_pos_weight,5
,verbosity,-1
,eval_metric,'logloss'


In [83]:
test_pred = model.predict_proba(X_test)[:, 1]
y_pred = model.predict_proba(X_train)[:, 1]

In [84]:
# Evaluation on training
training_cls_report = classification_report(
    y_train,
    y_pred > 0.5,
    target_names=["Not Stress", "Stress"],
)

print(f"Training:\n{training_cls_report}\n============\n")
# Evaluation on test set
cls_report = classification_report(
    y_test,
    test_pred > 0.5,
    target_names=["Not Stress", "Stress"],
)
print("Validation Metrics:\n", cls_report, end="\n\n")
print("ROC SCORE: ", roc_auc_score(y_test, test_pred))
print("LOG LOSS: ", log_loss(y_test, test_pred))


Training:
              precision    recall  f1-score   support

  Not Stress       0.98      0.87      0.92     27188
      Stress       0.55      0.88      0.68      4812

    accuracy                           0.88     32000
   macro avg       0.76      0.88      0.80     32000
weighted avg       0.91      0.88      0.89     32000


Validation Metrics:
               precision    recall  f1-score   support

  Not Stress       0.94      0.85      0.89      6812
      Stress       0.44      0.71      0.55      1188

    accuracy                           0.83      8000
   macro avg       0.69      0.78      0.72      8000
weighted avg       0.87      0.83      0.84      8000


ROC SCORE:  0.8661318360745843
LOG LOSS:  0.39957509979958983
